Legge `bronze_documenti_estratti` (una riga per PDF, con il risultato come JSON in una colonna stringa) e lo scompone in tabelle Silver tipizzate,
una per ciascun tipo di documento (gli ordini, le quotazioni e le
richieste di informazioni hanno schemi diversi tra loro, vedi
matching.py / quotazione.py / richiesta_informazioni.py nel notebook 01).

Tabelle prodotte:
- `silver_ordini`             - testata ordine (1 riga per documento)
- `silver_ordini_righe`       - righe ordine (1 riga per articolo ordinato)
- `silver_quotazioni`         - testata quotazione
- `silver_quotazioni_righe`   - righe quotazione
- `silver_richieste_info`     - richieste di informazioni (con articoli trovati)
- `silver_documenti_non_gestiti` - non_determinabile / errori tecnici

 Strategia di scrittura: overwrite completo a ogni esecuzione (ricostruisce
 Silver da zero leggendo tutta Bronze). Per dataset di queste dimensioni
 (decine/centinaia di documenti) e' la scelta piu' semplice e robusta;
 se in futuro Bronze cresce molto, si puo' passare a un merge incrementale
 basato su pdf_nome + data_elaborazione.


In [11]:
TABELLA_BRONZE = "bronze_documenti_estratti"
 
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, BooleanType, ArrayType
)
 
df_bronze = spark.sql(f"SELECT * FROM {TABELLA_BRONZE}")
print(f"Documenti totali in Bronze: {df_bronze.count()}")
df_bronze.groupBy("tipo_documento").count().show()

StatementMeta(, 033a172b-88ec-41fa-86da-29acc1f71fad, 13, Finished, Available, Finished, False)

Documenti totali in Bronze: 42
+-----------------+-----+
|   tipo_documento|count|
+-----------------+-----+
|           ordine|   40|
|non_determinabile|    1|
|       quotazione|    1|
+-----------------+-----+



In [12]:
# Parsing date robusto: l'agente puo' restituire date in formati diversi
# (es. "19/05/2026" italiano dd/MM/yyyy, "2026-05-19" ISO, o testo esteso
# tipo "giovedi' 21 maggio 2026 17:12" - riscontrato nei documenti di tipo
# email/richiesta informale). to_date() senza formato esplicito si aspetta
# solo yyyy-MM-dd e fallisce silenziosamente su qualsiasi altro formato.

def parsa_data_flessibile(nome_colonna: str):
    """Prova piu' formati di data in sequenza su una colonna (passata per
    nome, come stringa SQL). Implementata come UDF Python: try_to_date() non
    e' disponibile in questo runtime Spark, quindi una UDF e' la soluzione
    compatibile con qualsiasi versione.

    Gestisce sia formati numerici standard sia testo italiano esteso (con
    giorno della settimana e/o orario opzionali, es. 'giovedi' 21 maggio
    2026 17:12' -> estrae solo giorno/mese/anno via regex, ignorando il
    resto)."""
    from pyspark.sql.functions import udf
    from pyspark.sql.types import DateType
    from datetime import datetime
    import re

    MESI_ITALIANI = {
        "gennaio": 1, "febbraio": 2, "marzo": 3, "aprile": 4,
        "maggio": 5, "giugno": 6, "luglio": 7, "agosto": 8,
        "settembre": 9, "ottobre": 10, "novembre": 11, "dicembre": 12,
    }

    def _parsa(testo):
        if testo is None:
            return None
        testo = testo.strip()

        formati = ["%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y"]
        for formato in formati:
            try:
                return datetime.strptime(testo, formato).date()
            except (ValueError, AttributeError):
                continue

        match = re.search(r"(\d{1,2})\s+(\w+)\s+(\d{4})", testo, re.IGNORECASE)
        if match:
            giorno, mese_testo, anno = match.groups()
            mese_num = MESI_ITALIANI.get(mese_testo.lower())
            if mese_num:
                try:
                    return datetime(int(anno), mese_num, int(giorno)).date()
                except ValueError:
                    return None

        return None

    parsa_data_udf = udf(_parsa, DateType())
    return parsa_data_udf(F.col(nome_colonna))


StatementMeta(, 033a172b-88ec-41fa-86da-29acc1f71fad, 14, Finished, Available, Finished, False)

In [13]:
# Schema per documenti di tipo ORDINE (corrisponde all'output di valida_ordine
# nel notebook 01 / matching.py)
 
schema_riga_ordine = StructType([
    StructField("codice_dichiarato", StringType()),
    StructField("descrizione_dichiarata", StringType()),
    StructField("codice_articolo_match", StringType()),
    StructField("descrizione_match", StringType()),
    StructField("quantita", DoubleType()),
    StructField("unita_misura", StringType()),
    StructField("prezzo_dichiarato", DoubleType()),
    StructField("prezzo_listino_catalogo", DoubleType()),
    StructField("confidenza_match", StringType()),
    StructField("note_anomalia", StringType()),
])
 
schema_cliente = StructType([
    StructField("id_cliente_match", StringType()),
    StructField("ragione_sociale", StringType()),
    StructField("partita_iva", StringType()),
    StructField("indirizzo", StringType()),
    StructField("citta", StringType()),
    StructField("provincia", StringType()),
    StructField("email", StringType()),
    StructField("telefono", StringType()),
    StructField("fonte_cliente", StringType()),
])
 
schema_ordine = StructType([
    StructField("tipo_documento", StringType()),
    StructField("pdf_nome", StringType()),
    StructField("data_elaborazione", StringType()),
    StructField("intervento_umano_necessario", BooleanType()),
    StructField("motivo_intervento_umano", ArrayType(StringType())),
    StructField("cliente", schema_cliente),
    StructField("riferimento_ordine", StringType()),
    StructField("data_ordine", StringType()),
    StructField("data_consegna_richiesta", StringType()),
    StructField("condizioni_pagamento", StringType()),
    StructField("note_generali", StringType()),
    StructField("righe", ArrayType(schema_riga_ordine)),
])

StatementMeta(, 033a172b-88ec-41fa-86da-29acc1f71fad, 15, Finished, Available, Finished, False)

In [14]:
# SILVER: ordini (testata + righe esplose in tabella separata)
 
df_ordini_raw = (
    df_bronze
    .filter(F.col("tipo_documento") == "ordine")
    .withColumn("dati", F.from_json(F.col("json_completo"), schema_ordine))
)
 
silver_ordini = df_ordini_raw.select(
    F.col("pdf_nome"),
    F.col("dati.data_elaborazione").alias("data_elaborazione"),
    F.col("dati.intervento_umano_necessario").alias("intervento_umano_necessario"),
    F.col("dati.motivo_intervento_umano").alias("motivo_intervento_umano"),
    F.col("dati.cliente.id_cliente_match").alias("id_cliente"),
    F.col("dati.cliente.ragione_sociale").alias("cliente_ragione_sociale"),
    F.col("dati.cliente.partita_iva").alias("cliente_partita_iva"),
    F.col("dati.cliente.citta").alias("cliente_citta"),
    F.col("dati.cliente.provincia").alias("cliente_provincia"),
    F.col("dati.cliente.fonte_cliente").alias("cliente_fonte"),
    F.col("dati.riferimento_ordine").alias("riferimento_ordine"),
    parsa_data_flessibile("dati.data_ordine").alias("data_ordine"),
    F.col("dati.data_consegna_richiesta").alias("data_consegna_richiesta_testo"),
    parsa_data_flessibile("dati.data_consegna_richiesta").alias("data_consegna_richiesta"),
    F.col("dati.condizioni_pagamento").alias("condizioni_pagamento"),
    F.col("dati.note_generali").alias("note_generali"),
)
 
silver_ordini_righe = (
    df_ordini_raw
    .select(F.col("pdf_nome"), F.posexplode(F.col("dati.righe")).alias("posizione", "riga"))
    .select(
        F.col("pdf_nome"),
        F.col("posizione"),
        F.col("riga.codice_dichiarato").alias("codice_dichiarato"),
        F.col("riga.descrizione_dichiarata").alias("descrizione_dichiarata"),
        F.col("riga.codice_articolo_match").alias("codice_articolo_match"),
        F.col("riga.descrizione_match").alias("descrizione_match"),
        F.col("riga.quantita").alias("quantita"),
        F.col("riga.unita_misura").alias("unita_misura"),
        F.col("riga.prezzo_dichiarato").alias("prezzo_dichiarato"),
        F.col("riga.prezzo_listino_catalogo").alias("prezzo_listino_catalogo"),
        F.col("riga.confidenza_match").alias("confidenza_match"),
        F.col("riga.note_anomalia").alias("note_anomalia"),
        (F.col("riga.quantita") * F.col("riga.prezzo_dichiarato")).alias("importo_riga_calcolato"),
    )
)
 
silver_ordini.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_ordini")
silver_ordini_righe.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_ordini_righe")
 
print(f"silver_ordini: {silver_ordini.count()} righe")
print(f"silver_ordini_righe: {silver_ordini_righe.count()} righe")

StatementMeta(, 033a172b-88ec-41fa-86da-29acc1f71fad, 16, Finished, Available, Finished, False)

silver_ordini: 40 righe
silver_ordini_righe: 161 righe


In [15]:
# Schema per documenti di tipo QUOTAZIONE (corrisponde a prepara_quotazione)
 
schema_riga_quotazione = StructType([
    StructField("codice_dichiarato", StringType()),
    StructField("descrizione_dichiarata", StringType()),
    StructField("codice_articolo_match", StringType()),
    StructField("descrizione_match", StringType()),
    StructField("quantita", DoubleType()),
    StructField("unita_misura", StringType()),
    StructField("prezzo_unitario_listino", DoubleType()),
    StructField("importo_riga", DoubleType()),
    StructField("confidenza_match", StringType()),
])
 
schema_riga_non_quotabile = StructType([
    StructField("codice_dichiarato", StringType()),
    StructField("descrizione_dichiarata", StringType()),
    StructField("motivo", StringType()),
])
 
schema_quotazione = StructType([
    StructField("tipo_documento", StringType()),
    StructField("pdf_nome", StringType()),
    StructField("data_elaborazione", StringType()),
    StructField("intervento_umano_necessario", BooleanType()),
    StructField("motivo_intervento_umano", ArrayType(StringType())),
    StructField("cliente", schema_cliente),
    StructField("riferimento_richiesta", StringType()),
    StructField("data_richiesta", StringType()),
    StructField("note_generali", StringType()),
    StructField("righe_quotazione", ArrayType(schema_riga_quotazione)),
    StructField("righe_non_quotabili", ArrayType(schema_riga_non_quotabile)),
    StructField("totale_quotazione_stimato", DoubleType()),
])

StatementMeta(, 033a172b-88ec-41fa-86da-29acc1f71fad, 17, Finished, Available, Finished, False)

In [16]:
# SILVER: quotazioni (testata + righe esplose)
 
df_quotazioni_raw = (
    df_bronze
    .filter(F.col("tipo_documento") == "quotazione")
    .withColumn("dati", F.from_json(F.col("json_completo"), schema_quotazione))
)
 
silver_quotazioni = df_quotazioni_raw.select(
    F.col("pdf_nome"),
    F.col("dati.data_elaborazione").alias("data_elaborazione"),
    F.col("dati.cliente.id_cliente_match").alias("id_cliente"),
    F.col("dati.cliente.ragione_sociale").alias("cliente_ragione_sociale"),
    F.col("dati.cliente.fonte_cliente").alias("cliente_fonte"),
    F.col("dati.riferimento_richiesta").alias("riferimento_richiesta"),
    parsa_data_flessibile("dati.data_richiesta").alias("data_richiesta"),
    F.col("dati.note_generali").alias("note_generali"),
    F.col("dati.totale_quotazione_stimato").alias("totale_quotazione_stimato"),
)
 
silver_quotazioni_righe = (
    df_quotazioni_raw
    .select(F.col("pdf_nome"), F.posexplode(F.col("dati.righe_quotazione")).alias("posizione", "riga"))
    .select(
        F.col("pdf_nome"),
        F.col("posizione"),
        F.col("riga.codice_articolo_match").alias("codice_articolo_match"),
        F.col("riga.descrizione_match").alias("descrizione_match"),
        F.col("riga.quantita").alias("quantita"),
        F.col("riga.unita_misura").alias("unita_misura"),
        F.col("riga.prezzo_unitario_listino").alias("prezzo_unitario_listino"),
        F.col("riga.importo_riga").alias("importo_riga"),
        F.col("riga.confidenza_match").alias("confidenza_match"),
    )
)
 
silver_quotazioni.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_quotazioni")
silver_quotazioni_righe.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_quotazioni_righe")
 
print(f"silver_quotazioni: {silver_quotazioni.count()} righe")
print(f"silver_quotazioni_righe: {silver_quotazioni_righe.count()} righe")

StatementMeta(, 033a172b-88ec-41fa-86da-29acc1f71fad, 18, Finished, Available, Finished, False)

silver_quotazioni: 1 righe
silver_quotazioni_righe: 6 righe


In [17]:
# Schema per documenti di tipo RICHIESTA_INFORMAZIONI
 
schema_articolo_trovato = StructType([
    StructField("richiesto_come", StringType()),
    StructField("codice", StringType()),
    StructField("descrizione", StringType()),
    StructField("categoria", StringType()),
    StructField("unita_misura", StringType()),
    StructField("prezzo_listino", DoubleType()),
    StructField("confidenza_match", StringType()),
])
 
schema_richiesta_info = StructType([
    StructField("tipo_documento", StringType()),
    StructField("pdf_nome", StringType()),
    StructField("data_elaborazione", StringType()),
    StructField("cliente", schema_cliente),
    StructField("domanda_o_richiesta", StringType()),
    StructField("articoli_trovati_in_catalogo", ArrayType(schema_articolo_trovato)),
    StructField("articoli_non_trovati", ArrayType(StringType())),
    StructField("bozza_risposta", StringType()),
])

StatementMeta(, 033a172b-88ec-41fa-86da-29acc1f71fad, 19, Finished, Available, Finished, False)

In [18]:
# SILVER: richieste di informazioni
 
df_info_raw = (
    df_bronze
    .filter(F.col("tipo_documento") == "richiesta_informazioni")
    .withColumn("dati", F.from_json(F.col("json_completo"), schema_richiesta_info))
)
 
silver_richieste_info = df_info_raw.select(
    F.col("pdf_nome"),
    F.col("dati.data_elaborazione").alias("data_elaborazione"),
    F.col("dati.cliente.id_cliente_match").alias("id_cliente"),
    F.col("dati.cliente.ragione_sociale").alias("cliente_ragione_sociale"),
    F.col("dati.domanda_o_richiesta").alias("domanda_o_richiesta"),
    F.size(F.col("dati.articoli_trovati_in_catalogo")).alias("numero_articoli_trovati"),
    F.col("dati.articoli_non_trovati").alias("articoli_non_trovati"),
    F.col("dati.bozza_risposta").alias("bozza_risposta"),
)
 
silver_richieste_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_richieste_info")
print(f"silver_richieste_info: {silver_richieste_info.count()} righe")

StatementMeta(, 033a172b-88ec-41fa-86da-29acc1f71fad, 20, Finished, Available, Finished, False)

silver_richieste_info: 0 righe


In [19]:
# SILVER: documenti non determinabili / falliti tecnicamente (nessun parsing
# specifico necessario, lo schema e' minimo e comune)
 
silver_documenti_non_gestiti = (
    df_bronze
    .filter(F.col("tipo_documento") == "non_determinabile")
    .select(
        F.col("pdf_nome"),
        F.col("data_elaborazione"),
        F.get_json_object(F.col("json_completo"), "$.motivo_intervento_umano[0]").alias("motivo"),
    )
)
 
silver_documenti_non_gestiti.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_documenti_non_gestiti")
print(f"silver_documenti_non_gestiti: {silver_documenti_non_gestiti.count()} righe")

StatementMeta(, 033a172b-88ec-41fa-86da-29acc1f71fad, 21, Finished, Available, Finished, False)

silver_documenti_non_gestiti: 1 righe


In [20]:
print("\nCompletato. Tabelle Silver create/aggiornate:")
for nome_tabella in [
    "silver_ordini", "silver_ordini_righe",
    "silver_quotazioni", "silver_quotazioni_righe",
    "silver_richieste_info", "silver_documenti_non_gestiti",
]:
    print(f"  - {nome_tabella}")

StatementMeta(, 033a172b-88ec-41fa-86da-29acc1f71fad, 22, Finished, Available, Finished, False)


Completato. Tabelle Silver create/aggiornate:
  - silver_ordini
  - silver_ordini_righe
  - silver_quotazioni
  - silver_quotazioni_righe
  - silver_richieste_info
  - silver_documenti_non_gestiti


In [21]:
df = spark.sql("SELECT * FROM LakeHouse.dbo.silver_ordini LIMIT 1000")
display(df)

StatementMeta(, 033a172b-88ec-41fa-86da-29acc1f71fad, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d782e319-a12d-4030-9de3-0b4f8f1130bb)